# GitHub Actions — CI/CD Pipeline Patterns Deep Dive

> **Topic: Advanced workflow patterns for production CI/CD — reusable workflows, matrix builds, OIDC authentication, and pipeline optimization techniques.**

## Topics
1. OIDC Authentication — Eliminating Long-Lived Credentials
2. Matrix Builds — Testing Across Environments
3. Reusable Workflows — DRY Pipelines
4. Pipeline Performance Optimization
5. Deployment Strategies via GitHub Actions

In [ ]:
"""
OIDC JWT Token Simulator
========================
Demonstrates how GitHub Actions OIDC tokens work:
1. GitHub issues a signed JWT
2. AWS verifies the JWT against GitHub's OIDC endpoint
3. AWS issues temporary credentials
4. Workflow uses temp credentials
5. Credentials expire after the job

No AWS keys stored anywhere!
"""
import base64
import hashlib
import hmac
import json
import time
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Optional


@dataclass
class GitHubOIDCToken:
    """Represents a GitHub-issued OIDC JWT (simplified)."""
    # Standard JWT claims
    iss: str = "https://token.actions.githubusercontent.com"
    sub: str = "repo:org/repo:ref:refs/heads/main"
    aud: str = "sts.amazonaws.com"
    exp: int = 0
    iat: int = 0

    # GitHub-specific claims
    repository: str = "org/payment-service"
    repository_owner: str = "org"
    ref: str = "refs/heads/main"
    workflow: str = "CI/CD Pipeline"
    job_workflow_ref: str = "org/payment-service/.github/workflows/ci.yml@refs/heads/main"
    environment: Optional[str] = None
    actor: str = "karthik-dev"

    def __post_init__(self):
        now = int(time.time())
        self.iat = now
        self.exp = now + 600  # 10 minutes max lifetime

    def to_jwt(self, secret: str = "github-oidc-private-key") -> str:
        """Create a signed JWT (simplified — real GitHub uses RS256)."""
        header  = base64.urlsafe_b64encode(json.dumps({"typ": "JWT", "alg": "HS256"}).encode()).decode()
        payload = base64.urlsafe_b64encode(json.dumps({
            "iss": self.iss, "sub": self.sub, "aud": self.aud,
            "exp": self.exp, "iat": self.iat,
            "repository": self.repository,
            "repository_owner": self.repository_owner,
            "ref": self.ref, "workflow": self.workflow,
            "environment": self.environment, "actor": self.actor,
        }).encode()).decode()
        sig_input = f"{header}.{payload}"
        sig = base64.urlsafe_b64encode(
            hmac.new(secret.encode(), sig_input.encode(), hashlib.sha256).digest()
        ).decode()
        return f"{header}.{payload}.{sig}"


@dataclass
class AWSTemporaryCredentials:
    access_key_id: str
    secret_access_key: str
    session_token: str
    expiration: datetime
    role_arn: str

    def is_expired(self) -> bool:
        return datetime.now() > self.expiration


class OIDCAuthSimulator:
    """Simulates the OIDC authentication flow between GitHub Actions and AWS."""

    def __init__(self, allowed_repos: list[str], role_arn: str):
        self.allowed_repos = allowed_repos
        self.role_arn = role_arn

    def github_issues_token(self, repo: str, branch: str) -> GitHubOIDCToken:
        """Step 1: GitHub issues OIDC JWT at job start."""
        token = GitHubOIDCToken(
            repository=repo,
            repository_owner=repo.split("/")[0],
            ref=f"refs/heads/{branch}",
        )
        print(f"[GitHub] Issued OIDC JWT for {repo} (expires in 10 min)")
        return token

    def aws_assume_role(self, token: GitHubOIDCToken) -> AWSTemporaryCredentials:
        """Step 2: AWS STS verifies JWT and issues temp credentials."""

        # AWS verifies: token is from the expected GitHub OIDC provider
        if not token.iss.startswith("https://token.actions.githubusercontent.com"):
            raise PermissionError("Token not from GitHub OIDC provider")

        # AWS verifies: token is for an allowed repository
        if token.repository not in self.allowed_repos:
            raise PermissionError(f"Repository {token.repository} not in trust policy")

        # AWS verifies: token is not expired
        if token.exp < time.time():
            raise PermissionError("OIDC token has expired")

        # Issue temporary credentials (15 min - 12 hrs)
        creds = AWSTemporaryCredentials(
            access_key_id=f"ASIA{'X' * 16}",
            secret_access_key=hashlib.sha256(token.repository.encode()).hexdigest()[:40],
            session_token=hashlib.sha256(str(token.iat).encode()).hexdigest(),
            expiration=datetime.now() + timedelta(hours=1),
            role_arn=self.role_arn,
        )
        print(f"[AWS STS] Verified OIDC JWT. Issued temporary credentials.")
        print(f"[AWS STS] Role: {self.role_arn}")
        print(f"[AWS STS] Expires: {creds.expiration.strftime('%H:%M:%S')}")
        return creds


# Simulate the full OIDC flow
oidc_sim = OIDCAuthSimulator(
    allowed_repos=["org/payment-service", "org/user-service"],
    role_arn="arn:aws:iam::123456789:role/github-actions-deploy-prod",
)

print("=== GitHub Actions OIDC Authentication Flow ===")
print()

print("Step 1: GitHub Actions job starts...")
token = oidc_sim.github_issues_token("org/payment-service", "main")
jwt_str = token.to_jwt()
print(f"  JWT (first 60 chars): {jwt_str[:60]}...")

print("\nStep 2: Workflow calls AWS STS AssumeRoleWithWebIdentity...")
temp_creds = oidc_sim.aws_assume_role(token)

print(f"\nStep 3: Workflow uses temporary credentials to deploy...")
print(f"  Access Key: {temp_creds.access_key_id[:8]}...")
print(f"  Expired already: {temp_creds.is_expired()}")

print(f"\n✅ SECURITY ADVANTAGE:")
print(f"  - No AWS_ACCESS_KEY_ID stored in GitHub Secrets")
print(f"  - Credentials expire automatically (job end or 1hr)")
print(f"  - If job is compromised, attacker has ≤1hr window")
print(f"  - AWS CloudTrail shows: role assumed by GitHub workflow X")

print(f"\nTesting rejected repo...")
bad_token = oidc_sim.github_issues_token("attacker/malicious-repo", "main")
try:
    oidc_sim.aws_assume_role(bad_token)
except PermissionError as e:
    print(f"  ✅ REJECTED: {e}")

In [ ]:
"""
Pipeline Performance Optimizer
================================
Analyzes a pipeline configuration and suggests optimizations.
Models the key bottlenecks: sequential jobs, missing cache, large artifacts.
"""
from dataclasses import dataclass, field
from typing import List, Set, Dict
import math


@dataclass
class PipelineJob:
    name: str
    duration_minutes: float
    needs: List[str] = field(default_factory=list)
    has_cache: bool = False
    cache_hit_rate: float = 0.8  # 80% cache hit rate typically
    cacheable_duration: float = 0.0  # portion that can be cached

    @property
    def effective_duration(self) -> float:
        if self.has_cache and self.cacheable_duration > 0:
            uncacheable = self.duration_minutes - self.cacheable_duration
            cached = self.cacheable_duration * (1 - self.cache_hit_rate)
            return uncacheable + cached
        return self.duration_minutes


class PipelineOptimizer:
    def __init__(self, jobs: List[PipelineJob]):
        self.jobs = {j.name: j for j in jobs}

    def critical_path(self) -> List[str]:
        """Find the longest path through the DAG (bottleneck)."""
        memo = {}

        def longest(job_name: str) -> float:
            if job_name in memo:
                return memo[job_name]
            job = self.jobs[job_name]
            dep_max = max((longest(d) for d in job.needs), default=0)
            result = dep_max + job.effective_duration
            memo[job_name] = result
            return result

        return max(self.jobs.keys(), key=lambda j: longest(j))

    def total_sequential_time(self) -> float:
        return sum(j.duration_minutes for j in self.jobs.values())

    def total_wall_time(self) -> float:
        """Parallel execution time (longest critical path)."""
        memo = {}
        def longest(name):
            if name in memo:
                return memo[name]
            job = self.jobs[name]
            dep_max = max((longest(d) for d in job.needs), default=0)
            memo[name] = dep_max + job.effective_duration
            return memo[name]
        return max(longest(j) for j in self.jobs)

    def optimizations(self) -> List[str]:
        suggestions = []
        for job in self.jobs.values():
            if not job.has_cache and job.duration_minutes > 2:
                suggestions.append(
                    f"ADD CACHE to '{job.name}': {job.duration_minutes:.0f}min job without caching"
                )
            if job.duration_minutes > 15 and not job.needs:
                suggestions.append(
                    f"SPLIT '{job.name}': {job.duration_minutes:.0f}min job could be parallelized"
                )
        return suggestions


# Model a real pipeline: before and after optimization

print("=" * 60)
print("❌ BEFORE: Poorly optimized pipeline")
print("=" * 60)

before_jobs = [
    PipelineJob("lint",              duration_minutes=3.0, has_cache=False),
    PipelineJob("unit-tests",        duration_minutes=8.0, has_cache=False, needs=["lint"]),  # sequential!
    PipelineJob("integration-tests", duration_minutes=15.0, has_cache=False, needs=["unit-tests"]),
    PipelineJob("build",             duration_minutes=12.0, has_cache=False, needs=["integration-tests"]),
    PipelineJob("deploy",            duration_minutes=5.0, needs=["build"]),
]

before = PipelineOptimizer(before_jobs)
print(f"  Sequential time: {before.total_sequential_time():.0f}min")
print(f"  Wall time:       {before.total_wall_time():.0f}min")
print(f"  Suggestions:")
for s in before.optimizations():
    print(f"    • {s}")

print()
print("=" * 60)
print("✅ AFTER: Optimized pipeline")
print("=" * 60)

after_jobs = [
    # Gate 1: all parallel, with caching
    PipelineJob("lint",              duration_minutes=1.5, has_cache=True, cacheable_duration=1.0),
    PipelineJob("unit-tests",        duration_minutes=4.0, has_cache=True, cacheable_duration=2.0),  # parallel!
    PipelineJob("integration-tests", duration_minutes=15.0, has_cache=True, cacheable_duration=3.0),  # parallel!
    # Gate 2: build (waits for all gate 1 to pass)
    PipelineJob("build",             duration_minutes=5.0, has_cache=True, cacheable_duration=3.5,
                needs=["lint", "unit-tests", "integration-tests"]),
    PipelineJob("deploy",            duration_minutes=5.0, needs=["build"]),
]

after = PipelineOptimizer(after_jobs)
print(f"  Sequential time: {after.total_sequential_time():.0f}min")
print(f"  Wall time:       {after.total_wall_time():.1f}min")
print(f"  Remaining suggestions:")
optimizations = after.optimizations()
print(f"    {'None — pipeline is well-optimized!' if not optimizations else chr(10).join(f'    • {s}' for s in optimizations)}")

speedup = before.total_wall_time() / after.total_wall_time()
print(f"\n  Wall time reduction: {before.total_wall_time():.0f}min → {after.total_wall_time():.1f}min")
print(f"  Speedup: {speedup:.1f}× faster")
print(f"  For a team of 20 engineers, 5 pushes/day each:")
saved_min = (before.total_wall_time() - after.total_wall_time()) * 20 * 5
print(f"    Time saved: {saved_min:.0f} min/day = {saved_min/60:.1f} engineer-hours/day")